In [ ]:
import random
import copy
import torch
import pickle
import os
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import *
from causal_rl.algo.imitation.gail.causal_gail import *

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '5'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
num_steps = 2000
confound_strength = 0.5
seed = 445
lookback = 2
hidden_dims = {'C'}

random.seed(seed)
torch.manual_seed(seed)

In [ ]:
# for training: regular W, O hidden
train_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed, confound_strength=confound_strength)

# for eval: corrupted W, O hidden
eval_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=False, seed=seed, confound_strength=confound_strength)

## Causal Graph Analysis

In [ ]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = HumanoidMazePCH(num_steps=small_steps, seed=seed, confound_strength=confound_strength)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [ ]:
naive_Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')
    naive_Z_sets[Xi] = cond

naive_Z_sets['X1']

## Expert Trajectories

In [ ]:
# for eval: corrupted W, O shown
expert_traj_env = HumanoidMazePCH(num_steps=num_steps, expert_mode=True, confound_strength=confound_strength)
# load model
MODEL_PATH = '/home/et2842/causal/causalrl/models/humanoidmaze_medium_expert_finetuned.pt'
expert_ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

expert_action_bounds = (expert_ckpt['action_bounds_low'], expert_ckpt['action_bounds_high'])

expert_model = ContinuousPolicyNN(
    input_dim=expert_ckpt['input_dim'],
    action_dim=expert_ckpt['num_actions'],
    hidden_dim=256,
    num_blocks=expert_ckpt['num_blocks'],
    dropout=expert_ckpt['dropout'],
    layernorm=expert_ckpt['layernorm'],
    final_tanh=expert_ckpt['final_tanh'],
    action_bounds=expert_action_bounds,
).to(device)

expert_model.load_state_dict(expert_ckpt['state_dict'])
expert_model.eval()

expert_slots = expert_ckpt['slots']
expert_Z_trim = expert_ckpt['Z_trim']
expert_dims = expert_ckpt['dims']
expert_lookback = expert_ckpt['lookback']

expert_policy = shared_policy_fn_long_horizon(expert_model, expert_slots, expert_Z_trim, continuous=True, device=device)
expert_policies = make_shared_policy_dict(expert_policy)
expert_num_eval_eps = 500

records = collect_imitator_trajectories(
    env=expert_traj_env,
    policies=expert_policies,
    num_episodes=expert_num_eval_eps,
    max_steps=num_steps,
    show_progress=True
)

len(records)

In [ ]:
dims = {
    'P': 2,
    'A': 21,
    'H': 1,
    'E': 12,
    'V': 3,
    # 'C': 3,
    'J': 27,
    'W': 2,
    'X': 21
}

In [ ]:
sample_obs = records[0]['obs']

# Trim Z-sets to the lookback window (this matches what you do for BC)
naive_Z_trim = trim_Z_sets(naive_Z_sets, lookback=lookback)

# Build windowed encoders that depend on relative lags (not absolute time)
naive_encode, naive_z_dim, naive_slots = build_windowed_z_encoder(
    naive_Z_trim,
    dims=dims,
    lookback=lookback,
)

naive_z_dim

In [ ]:
# precompute expert batches once (so one_training_round doesn't redo this every time)
Z_e_naive, A_e_naive, X_e_naive = make_expert_batch(records, naive_encode)
X_e_naive = X_e_naive.to(device)

## Hyperparameters

In [ ]:
# PPO
gail_gamma          = 0.99
gae_lambda          = 0.95
ppo_clip            = 0.2
ppo_epochs          = 2
ppo_minibatch_size  = 1024
entropy_coeff       = 1e-2
value_coeff         = 0.5
max_grad_norm       = 0.5
normalize_adv       = True

# discriminator
d_loss_type         = 'bce'
gp_lambda           = 5.0
d_updates           = 2
d_minibatch_size    = 1024
use_gp              = True
instance_noise_std  = 0.0
label_smoothing     = 0.0

# rollout
max_steps_per_episode   = num_steps
episodes_per_round      = 10
num_rounds_naive_gail  = 200

# network
hidden_size_actor   = 256
hidden_size_critic  = 256
hidden_size_disc    = 256
actor_lr            = 1e-4
critic_lr           = 3e-4
disc_lr             = 3e-4
num_blocks_actor    = 3
dropout_actor       = 0.05
layernorm_actor     = True

## Network Initialization

In [ ]:
action_dim = train_env.env.action_space.shape[0]
action_low = float(train_env.env.action_space.low.min())
action_high = float(train_env.env.action_space.high.max())

naive_actor = ContinuousActor(
    num_inputs=naive_z_dim,
    num_outputs=action_dim,
    hidden_size=hidden_size_actor,
    std=0.0,
    action_low=action_low,
    action_high=action_high,
    num_blocks=num_blocks_actor,
    dropout=dropout_actor,
    layernorm=layernorm_actor,
).to(device)

naive_critic = Critic(
    num_inputs=naive_z_dim,
    hidden_size=hidden_size_critic,
).to(device)

naive_disc = Discriminator(
    num_inputs=naive_z_dim + action_dim,
    hidden_size=hidden_size_disc,
    dropout=0.2,
).to(device)

actor_optim_naive = torch.optim.Adam(naive_actor.parameters(), lr=actor_lr)
critic_optim_naive = torch.optim.Adam(naive_critic.parameters(), lr=critic_lr)
disc_optim_naive = torch.optim.Adam(naive_disc.parameters(), lr=disc_lr)

## Training

In [ ]:
best_return = -float('inf')
best_actor_sd = None
return_window = []
WINDOW = 20

disc_scheduler = torch.optim.lr_scheduler.StepLR(disc_optim_naive, step_size=100, gamma=0.5)

logs_naive_gail = []

for it in range(1, num_rounds_naive_gail + 1):
    stats = one_training_round(
        env=train_env,
        actor=naive_actor,
        critic=naive_critic,
        discriminator=naive_disc,
        actor_optim=actor_optim_naive,
        critic_optim=critic_optim_naive,
        discriminator_optim=disc_optim_naive,
        encode=naive_encode,
        X_e=X_e_naive,
        expert_records=None,
        gamma=gail_gamma,
        gae_lambda=gae_lambda,
        ppo_clip=ppo_clip,
        epochs=ppo_epochs,
        minibatch_size=ppo_minibatch_size,
        entropy_coeff=entropy_coeff,
        value_coeff=value_coeff,
        max_grad_norm=max_grad_norm,
        normalize_adv=normalize_adv,
        loss_type=d_loss_type,
        gp_lambda=gp_lambda,
        d_updates=d_updates,
        d_minibatch_size=d_minibatch_size,
        use_gp=use_gp,
        instance_noise_std=instance_noise_std,
        label_smoothing=label_smoothing,
        max_steps=max_steps_per_episode,
        num_episodes=episodes_per_round,
        seed=seed + 10_000 + it
    )
    logs_naive_gail.append(stats)
    disc_scheduler.step()

    # rolling average return tracking
    return_window.append(stats['avg_env_return'])
    if len(return_window) > WINDOW:
        return_window.pop(0)
    avg_ret = sum(return_window) / len(return_window)

    if avg_ret > best_return:
        best_return = avg_ret
        best_actor_sd = copy.deepcopy(naive_actor.state_dict())

    if it % 10 == 0:
        print(
            f"[Naive GAIL iter {it}] "
            f"return={stats['avg_env_return']:.2f}, "
            f"D_loss={stats['D_loss']:.3f}, "
            f"actor_loss={stats['ppo_actor_loss']:.3f}, "
            f"best_avg={best_return:.2f}"
        )

# restore best checkpoint
naive_actor.load_state_dict(best_actor_sd)
print(f"Restored best checkpoint (avg return={best_return:.2f})")

In [ ]:
naive_gail_policy = make_gail_policy(naive_actor, naive_encode, device=device, deterministic=True)
naive_gail_policies = make_shared_policy_dict(naive_gail_policy)

## Save Model

In [ ]:
# save model
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, 'ngail_hummed_c50_s445.pt')

naive_gail_ckpt = {
    "state_dict": naive_actor.state_dict(),
    "z_dim": naive_z_dim,
    "action_dim": action_dim,
    "hidden_size_actor": hidden_size_actor,
    "num_blocks_actor": num_blocks_actor,
    "dropout_actor": dropout_actor,
    "layernorm_actor": layernorm_actor,
    "final_tanh": True,
    "action_bounds_low": eval_env.env.action_space.low,
    "action_bounds_high": eval_env.env.action_space.high,
    "Z_sets": naive_Z_trim,
    "dims": dims,
    "lookback": lookback,
}

torch.save(naive_gail_ckpt, MODEL_PATH)
print("Saved Naive GAIL actor to:", MODEL_PATH)

## Evaluation

In [ ]:
num_eval_eps = 500
ngail_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=naive_gail_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(ngail_returns)

In [ ]:
ngail_episode_rewards = defaultdict(float)
for rec in ngail_returns:
    ep = rec['episode']
    ngail_episode_rewards[ep] += float(rec['reward'])

ngail_rewards = [ngail_episode_rewards[e] for e in range(num_eval_eps)]
sum(ngail_rewards) / num_eval_eps

In [ ]:
mean_reward = np.mean(ngail_rewards)
std_reward = np.std(ngail_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

In [ ]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in ngail_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")